# Step 0 — LightGBM 기준선 OOF 재생성 (Baseline OOF Regeneration)

- **목적 (Objective)**: `baseline[Lab].ipynb` 가 메모리에만 두고 버린 **Out-of-Fold 예측을 파일로 복원**한다.
- **배경 (Rationale)**: 기준 노트북은 `test_pred` 만 `submission.csv` 로 저장하고 `oof_pred` 는 저장하지 않는다.
  스태킹 메타 모델은 베이스 모델의 OOF 예측을 입력으로 요구하므로, 이 파일이 없으면 앙상블 단계 자체가 성립하지 않는다.
- **원칙 (Constraint)**: 성능 개선이 아닌 **파일 복구**가 목적이므로 `LGB_PARAMS`, `SEED`, `N_FOLDS`,
  `num_boost_round`, `early_stopping` 을 기준 노트북과 **완전히 동일하게** 유지한다.

| 생성 산출물 | 스키마 | 용도 |
|---|---|---|
| `outputs/oof_lgbm.csv` | `id, fold, pred` × 699,635 | 스태킹 메타 모델 학습 입력 |
| `outputs/test_lgbm.csv` | `id, pred` × 299,844 | 스태킹 결과의 Test 추론 |

---

## 1. 환경 설정 및 데이터 경로 탐색 (Environment & Path Setup)

- **재현성 (Reproducibility)**: 전역 시드 `SEED = 42` 고정.
- **경로 자동 해결**: `find_data_dir()` 가 `data/`, `../data/`, Kaggle `/kaggle/input` 을 순차 탐색하므로
  본 노트북이 `chacha/` 하위에 있어도 상위의 `../data/` 를 그대로 참조한다.

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
START_TIME = time.time()

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("lightgbm    ", lgb.__version__)

pandas       3.0.5
numpy        2.5.3
scikit-learn 1.9.1
lightgbm     4.7.0


In [2]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)


def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.")
    raise RuntimeError("데이터 폴더 후보가 여러 개입니다:\n" + "\n".join(f"  - {c}" for c in candidates))


DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR)
print("OUT_DIR  =", OUT_DIR.resolve())

DATA_DIR = /Users/chanjong/sesac-ai/중간평가/data
OUT_DIR  = /Users/chanjong/sesac-ai/중간평가/chacha/outputs


## 2. 데이터 적재 및 전처리 (Data Ingestion & Preprocessing)

기준 노트북 cell 5 / 8 / 28 과 동일한 로직:
- 타깃 이진화: Boolean → Integer
- 범주형 4종을 **train 기준 범주 집합**으로 `pd.Categorical` 캐스팅 (LightGBM native categorical 활용)
- 식별자 `id` 제거, 결측(`Arrival Delay in Minutes`)은 LightGBM 분기 학습을 위해 **NaN 유지**

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
NUM_COLS = [c for c in train.columns if c not in CAT_COLS + ["id", TARGET]]
RATING_COLS = [c for c in NUM_COLS if train[c].max() <= 5]

y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)
    assert X[col].notna().all() and X_test[col].notna().all(), f"미등록 범주 발생: {col}"

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

X      : (699635, 21)
X_test : (299844, 21)
y mean : 0.4436


## 3. Fold 계약 및 산출물 저장 규약 (Fold Contract & Artifact Convention)

- **Fold 계약 (§3-1)**: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` 를 **문자 그대로** 사용.
  `StratifiedKFold.split` 은 `X` 의 내용이 아닌 **샘플 수와 `y`** 만 참조하므로,
  기준 노트북(`X` 전달)과 MLP 노트북(`train` 전달)의 분할은 **수학적으로 동일**하다.
- **`fold_id` 는 0-based**: MLP 노트북이 생성한 `oof_mlp.csv` 의 `fold` 컬럼과 동일한 체계를 유지해야 병합 검증이 성립한다.
  (기준 노트북의 출력 로그는 `start=1` 로 1-based 표기하지만, 이는 출력 문자열에만 영향을 준다.)

In [4]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

fold_id = np.full(len(train), -1)
for k, (_, va_idx) in enumerate(FOLDS):   # 0-based: oof_mlp.csv 와 동일 체계
    fold_id[va_idx] = k
assert (fold_id >= 0).all()
print("fold sizes:", np.bincount(fold_id))


def save_predictions(name: str, oof: np.ndarray, test_pred: np.ndarray) -> None:
    """OOF / Test 예측을 id 기준으로 저장해 스태킹 단계에서 안전하게 정렬 가능하도록 한다."""
    assert np.isfinite(oof).all() and np.isfinite(test_pred).all()
    pd.DataFrame({"id": train["id"].values, "fold": fold_id, "pred": oof}).to_csv(OUT_DIR / f"oof_{name}.csv", index=False)
    pd.DataFrame({"id": test["id"].values, "pred": test_pred}).to_csv(OUT_DIR / f"test_{name}.csv", index=False)
    print(f"saved oof_{name}.csv / test_{name}.csv")

fold sizes: [139927 139927 139927 139927 139927]


## 4. LightGBM 5-Fold 학습 (Identical Re-run)

기준 노트북 cell 33과 **동일한 하이퍼파라미터**. 변경점 없음 — 재현성 검증이 목적이다.

- Fold별 Gain 중요도는 **각 Fold 내에서 합이 1이 되도록 정규화한 뒤** 평균한다.
  Fold별 트리 수가 256~549개로 2배 이상 차이나므로, 원시 Gain을 그대로 평균하면 트리가 많은 Fold가 과대 반영된다.
  (기준 노트북 cell 37의 보정 사항)

In [5]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": SEED,
    "verbose": -1,
}

oof_pred = np.full(len(X), np.nan)
oof_count = np.zeros(len(X), dtype=int)
test_pred = np.zeros(len(X_test))
importance = pd.Series(0.0, index=X.columns)
fold_aucs, best_iters = [], []

for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
    dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=5000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
    oof_count[va_idx] += 1
    test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

    gain = pd.Series(model.feature_importance(importance_type="gain"), index=X.columns)
    importance += gain / gain.sum() / N_FOLDS   # fold별 정규화 후 평균

    fold_auc = roc_auc_score(y.iloc[va_idx], oof_pred[va_idx])
    fold_aucs.append(fold_auc)
    best_iters.append(model.best_iteration)
    print(f"Fold {fold + 1}: AUC = {fold_auc:.5f}   trees = {model.best_iteration}   ({time.time() - START_TIME:.0f}s)")

assert (oof_count == 1).all()
assert np.isfinite(oof_pred).all()

lgb_auc = roc_auc_score(y, oof_pred)
print(f"\nLightGBM OOF AUC = {lgb_auc:.5f}   accuracy = {accuracy_score(y, oof_pred > 0.5):.5f}")
print(f"fold AUC  mean = {np.mean(fold_aucs):.5f}   std = {np.std(fold_aucs):.5f}")
save_predictions("lgbm", oof_pred, test_pred)

Fold 1: AUC = 0.95877   trees = 549   (12s)


Fold 2: AUC = 0.95759   trees = 293   (19s)


Fold 3: AUC = 0.95930   trees = 286   (26s)


Fold 4: AUC = 0.95870   trees = 280   (33s)


Fold 5: AUC = 0.95902   trees = 256   (39s)

LightGBM OOF AUC = 0.95866   accuracy = 0.92590
fold AUC  mean = 0.95868   std = 0.00058


saved oof_lgbm.csv / test_lgbm.csv


## 5. 재현성 검증 (Reproduction Checks) ★ 본 단계의 핵심 산출물

세 가지를 확인한다. 하나라도 실패하면 Fold 계약이 깨진 것이므로 Step 1 진행을 중단한다.

1. **OOF AUC 일치**: 기준선 `0.95866` 대비 오차 ≤ `0.0002`
2. **Fold 배정 일치**: `oof_lgbm.csv` 와 `oof_mlp.csv` 의 `fold` 컬럼이 `id` 기준 병합 시 전 행 동일
   — 위치 비교가 아닌 **`id` 조인**으로 검증해, 두 파일의 행 순서 가정에 의존하지 않는다.
3. **Test 예측 일치**: 기존 `submission.csv` 와 최대 절대 오차 < `1e-9`

In [6]:
BASELINE_AUC = 0.95866
REPORT = {}

# (1) OOF AUC 재현
delta = lgb_auc - BASELINE_AUC
REPORT["oof_auc"] = (lgb_auc, delta, abs(delta) <= 2e-4)
print(f"(1) OOF AUC  {lgb_auc:.5f}  (baseline {BASELINE_AUC}, Δ={delta:+.5f})  -> {'PASS' if abs(delta) <= 2e-4 else 'FAIL'}")

# (2) fold 배정이 MLP 노트북과 동일한가 (id 조인 기준)
mlp_path = OUT_DIR / "oof_mlp.csv"
if mlp_path.exists():
    a = pd.read_csv(OUT_DIR / "oof_lgbm.csv", usecols=["id", "fold"])
    b = pd.read_csv(mlp_path, usecols=["id", "fold"])
    m = a.merge(b, on="id", suffixes=("_lgbm", "_mlp"), validate="one_to_one")
    ok = len(m) == len(train) and (m["fold_lgbm"] == m["fold_mlp"]).all()
    REPORT["fold_contract"] = ok
    print(f"(2) fold 계약 일치  rows={len(m):,}  -> {'PASS' if ok else 'FAIL'}")
    assert ok, "Fold 배정 불일치 — 스태킹 불가. 원인 규명 전까지 중단."
else:
    print("(2) oof_mlp.csv 없음 — 건너뜀")

# (3) 기존 submission.csv 와 test 예측 일치
sub_path = Path("../submission.csv")
if sub_path.exists():
    old = pd.read_csv(sub_path)
    new = pd.read_csv(OUT_DIR / "test_lgbm.csv").rename(columns={"pred": TARGET})
    cmp = old.merge(new, on="id", suffixes=("_old", "_new"), validate="one_to_one")
    max_err = (cmp[f"{TARGET}_old"] - cmp[f"{TARGET}_new"]).abs().max()
    REPORT["test_match"] = (max_err, max_err < 1e-9)
    print(f"(3) test 예측 일치  max|Δ| = {max_err:.3e}  -> {'PASS' if max_err < 1e-9 else 'FAIL'}")
else:
    print("(3) ../submission.csv 없음 — 건너뜀")

(1) OOF AUC  0.95866  (baseline 0.95866, Δ=+0.00000)  -> PASS
(2) fold 계약 일치  rows=699,635  -> PASS


(3) test 예측 일치  max|Δ| = 0.000e+00  -> PASS


## 6. Fold별 성능 분포 (Per-Fold Score Distribution)

§3-4 판정 기준을 위해 **Fold별 AUC 5개를 모두 기록**한다.
Fold 간 표준편차(≈0.0006)가 목표 개선폭(+0.0013)과 같은 자릿수이므로,
이후 실험은 Pooled OOF 단일 값이 아니라 **5개 Fold 중 4개 이상에서 개선**되어야 유효한 것으로 간주한다.

In [7]:
fold_table = pd.DataFrame({
    "fold": range(1, N_FOLDS + 1),
    "auc": fold_aucs,
    "trees": best_iters,
    "n_valid": [len(va) for _, va in FOLDS],
})
print(fold_table.round(5).to_string(index=False))
print(f"\npooled OOF AUC = {lgb_auc:.5f}")
fold_table.to_csv(OUT_DIR / "folds_lgbm.csv", index=False)

print("\n--- Top 10 feature importance (fold-normalised gain) ---")
print((importance.sort_values(ascending=False).head(10) * 100).round(2).to_string())
print(f"\n전체 실행 시간: {time.time() - START_TIME:.0f}초")

 fold     auc  trees  n_valid
    1 0.95877    549   139927
    2 0.95759    293   139927
    3 0.95930    286   139927
    4 0.95870    280   139927
    5 0.95902    256   139927

pooled OOF AUC = 0.95866

--- Top 10 feature importance (fold-normalised gain) ---
Online boarding           52.01
Inflight wifi service     13.05
Type of Travel            12.51
Customer Type              3.83
Inflight entertainment     3.65
Class                      3.21
Checkin service            2.24
Baggage handling           1.54
On-board service           1.31
Seat comfort               1.25

전체 실행 시간: 40초
